In [1]:
import services.utils as ut
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from nltk.stem import WordNetLemmatizer
from gensim.parsing.preprocessing import STOPWORDS
from gensim.utils import simple_preprocess
import random
import services.utils as ut
import services.model as md
import joblib
import json
np.random.seed(42)
random.seed(42)

In [2]:
def predictionoflabels(predictions, jsonfile):
    with open(jsonfile, 'r') as f:
        labels_dict = json.load(f)
    for pred in predictions:
        label_name = labels_dict.get(str(pred), "Unknown Cluster")
        print(f"Cluster {pred}: {label_name}")

In [3]:
custom_words = {
    'please', 'help', 'assist', 'support', 'thanks', 'thank','soon','mentioned',
    'im', 'ive', 'us','would', 'could', 'need', 'want', 'trying',
    'tried','check', 'checked', 'make', 'made', 'get', 'getting','also',
    'use', 'using', 'used','thing', 'something', 'anything', 'everything',
    'way', 'time','issue', 'problem', 'request', 'work', 'working', 'fine',
    'available', 'recent', 'recently','facing', 'doe', 'noticed', 'happening',
    'started', 'happen','different', 'steps', 'did', 'regards','already', 'multiple',
    'last','times','followed', 'reviewed','specific', 'possible', 'related','new',
    'old','find', 'try', 'say', 'mean','name', 'email', 'price', 'one', 'unresolved',
    'add','note', 'may', 'dont', 'know','sure', 'changes', 'performed', 'properly',
    'original','like', 'similar','reported','doesnt', 'sometimes', 'acts', 'works',
    'ensure', 'desired', 'action', 'remains', 'life', 'seems', 'might', 'guide',
    'much', 'others', 'heavily', 'daily', 'task', 'affecting', 'assistance','hoping',
    'persists','didnt','option', 'perform', 'recommendation', 'information', 'official',
    'solution', 'provide', 'making', 'user', 'customer', 'item', 'device','far', 'luck',
    'contact', 'contacted', 'occurring','resolve', 'function', 'came', 'having', 'change',
    'haven', 'let', 'unable', 'able', 'afterward', 'var', 'step', 'order'
}



In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\{.*?\}|\[.*?\]|<.*?>|\(.*?\)', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text) # remove URLs
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', ' ', text) # remove email addresses
    text = re.sub(r'[^\x00-\x7F]+', ' ', text) # remove non ASCII characters
    text = re.sub(r'[^a-z\s]', '', text) 
    text = re.sub(r'\b[a-zA-Z]{1,2}\b', '', text)  # remove very short words
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [5]:
lemmatizer = WordNetLemmatizer()
final_stopwords = STOPWORDS.union(custom_words)
custom_words_lemma = set([lemmatizer.lemmatize(w.lower()) for w in final_stopwords])

def preprocess(text):
    text = str(text).lower()
    tokens = simple_preprocess(text, deacc=True)
    processed_tokens = []
    for word in tokens:
        lemma = lemmatizer.lemmatize(word)
        if (lemma not in custom_words_lemma and len(lemma) > 2 and lemma.isalnum()):
            processed_tokens.append(lemma)
    
    return " ".join(processed_tokens)

In [6]:
def nlp_cleaner(text_list):
    results = []
    for text in text_list:
        cleaned = clean_text(text) 
        processed = preprocess(cleaned)
        results.append(processed)
    return results

In [7]:
def clean_for_embeddings(text):
    text = text.lower()
    text = re.sub(r'\{.*?\}|\[.*?\]|<.*?>|\(.*?\)', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text) # remove URLs
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', ' ', text) # remove email addresses
    text = re.sub(r'[^\x00-\x7F]+', ' ', text) # remove non-ASCII characters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [8]:
def nlp_cleaner_embeddings(text_list):
    results = []
    for text in text_list:
        cleaned = clean_for_embeddings(text) 
        results.append(cleaned)
    return results

In [41]:
raw_data = ["I need help with my account setup", "Software is crashing in the production environment"]

## Kmeans

### Kmean TFIDF without SVD

In [45]:
Kmeans_tfidf_without_svd = ut.load_model('kmeans', 'kmeans_tfidf_without_pca.joblib')
predictions = Kmeans_tfidf_without_svd.predict(raw_data)
Kmeans_tfidf_without_svd_prediction_file = ut.loadingfile('kmeans', 'kmeanstfidfwithoutsvd_cluster_labels.json')
predictionoflabels(predictions, Kmeans_tfidf_without_svd_prediction_file)

Cluster 1: General Productivity & Power Config
Cluster 3: Software Version & Crash Reports


### Kmean TFIDF with SVD

In [47]:
kmeans_tfidf_svd = ut.load_model('kmeans', 'kmeans_tfidf_with_svd.joblib')
kmeans_tfidf_svd_prediction_file = ut.loadingfile('kmeans', 'kmeanstfidfsvd_cluster_labels.json')
predictions = kmeans_tfidf_svd.predict(raw_data)
predictionoflabels(predictions, kmeans_tfidf_svd_prediction_file)

Cluster 1: Admin & Account Locking
Cluster 11: Online Profile Authorization


### Kmean Embeddings without PCA

In [48]:
kmeans_embeddings_without_pca = ut.load_model('kmeans', 'kmeans_embeddings_without_pca.joblib')
kmeans_embeddings_without_pca_prediction_file = ut.loadingfile('kmeans', 'kmeansembeddingswithoutpca_cluster_labels.json')
predictions = kmeans_embeddings_without_pca.predict(raw_data)
predictionoflabels(predictions, kmeans_embeddings_without_pca_prediction_file)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Cluster 7: Credential Errors & Widespread Device Issues
Cluster 4: Software Bugs & Application Crashes


### Kmean Embeddings with PCA

In [49]:
kmeans_embeddings_with_pca = ut.load_model('kmeans', 'kmeans_embeddings_with_pca.joblib')
kmeans_embeddings_with_pca_prediction_file = ut.loadingfile('kmeans', 'kmeansembeddingswithpca_cluster_labels.json')
predictions = kmeans_embeddings_with_pca.predict(raw_data)
predictionoflabels(predictions, kmeans_embeddings_with_pca_prediction_file)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Cluster 10: Data Security & Privacy Concerns
Cluster 9: Application Crashes & Software Bugs


## LDA

### LDA with TFIDF

In [57]:
ladtfidf = ut.load_model('lda', 'lda_with_tfidf.joblib')
ladtfidf_prediction_file = ut.loadingfile('lda', 'LDAtfidf_pipeline_cluster_labels.json')
predictionsldatfidf = ladtfidf.transform(raw_data)
predictions_final_ldatfidf = np.argmax(predictionsldatfidf, axis=1)
predictionoflabels(predictions_final_ldatfidf, ladtfidf_prediction_file)

Cluster 1: Community Tutorials & Online Troubleshooting
Cluster 12: Software Version Updates & Maintenance


### LDA with countvectorizer

In [58]:
ldacountvectorizer = ut.load_model('lda', 'lda_with_count_vectorizer.joblib')
ldacountvectorizer_prediction_file = ut.loadingfile('lda', 'ladwithcountvectorizer_pipeline_cluster_labels.json')
predictionsldacountvectorizer = ldacountvectorizer.transform(raw_data)
predictions_final_ldacountvectorizer = np.argmax(predictionsldacountvectorizer, axis=1)
predictionoflabels(predictions_final_ldacountvectorizer, ldacountvectorizer_prediction_file)

Cluster 1: Account Lockout & Unlock Assistance
Cluster 3: Software Updates & Application Crashes


### LDA with Gensim Topics

In [64]:
def predict_gensim_lda(model, dictionary, raw_docs):
    tokenized_data = [str(doc).split() for doc in raw_docs]
    bow_corpus = [dictionary.doc2bow(text) for text in tokenized_data]
    final_labels = []
    for bow in bow_corpus:
        topic_probs = model.get_document_topics(bow, minimum_probability=0.0)
        best_topic = max(topic_probs, key=lambda x: x[1])[0]
        final_labels.append(best_topic)
        
    return np.array(final_labels)

In [65]:
ldagensim = ut.load_model('lda', 'lda_gensim_topics.joblib')
dictionarygensim = ut.load_model('lda', 'lda_gensim_topics_dict.joblib')
ldagensim_prediction_file = ut.loadingfile('lda', 'lda_gensim_topics_pipeline_cluster_labels.json')
predictions_final = predict_gensim_lda(ldagensim, dictionarygensim, raw_data)
predictionoflabels(predictions_final, ldagensim_prediction_file)

Cluster 16: Accidental Deletion & Urgent Data Recovery
Cluster 0: Software Version & Payment Inquiries
